In [ ]:

from types import SimpleNamespace
import numpy as np
import math

class AM81111:

    encoderUnit = SimpleNamespace(**{ 
        'bits': [16, 16]
    })

    driveUnit = SimpleNamespace(**{ 
        'gearBoxGearRatio': 1000.0,        
        'timingBeltTransmissionGearRatio': 2.0,
        'spindlePitch': 5.0,
        'motorIncrementPositions': 262_144, 
        'cylinderDiameter': 15.0,
        'limit': {
            'low': 0,
            'high': 24_185_993 
        }
    })

    cylinderUnit = SimpleNamespace(**{ 
        'limit': {
            'low':   81.3,  # mm bottom
            'high': 131.1   # mm top
        }
    })

    def gearRatio(self):
        return self.driveUnit.timingBeltTransmissionGearRatio * self.driveUnit.gearBoxGearRatio

    def cylinderArea(self):
        return np.pow(self.driveUnit.cylinderDiameter,2) * np.pi / 4.
    
    def pitchVolume(self):
        return self.driveUnit.spindlePitch * self.cylinderArea()

    def mulmin2incs(self, value):
        # mm/r
        transmission = self.driveUnit.spindlePitch / self.gearRatio()
        # µl/r
        injectionRateRotation = transmission * self.cylinderArea()
        # µl/inc
        injectionRateIncrement = injectionRateRotation / self.driveUnit.motorIncrementPositions
        # µl/min
        return np.clip(value / (injectionRateIncrement * 60), self.driveUnit.limit['low'], self.driveUnit.limit['high'])
    
    def incs2mulmin(self, value):
        # mm/r
        transmission = self.driveUnit.spindlePitch / self.gearRatio()
        # mm³/r ~ µl/r
        injectionRateRotation = transmission * self.cylinderArea()
        # µl/inc
        injectionRateIncrement = injectionRateRotation / self.driveUnit.motorIncrementPositions
        return (value * injectionRateIncrement * 60.0)
    
    def mms2mulmin(self, value):
        # mm³/s ~ µl/s
        return value * self.cylinderArea() * 60
    
    def turn2ml(self, value):

        bitRange = 2 ** self.encoderUnit.bits[1] -1
        
        mtb = value[0] * self.pitchVolume() / self.gearRatio()
        stb = value[1] / bitRange * self.pitchVolume() / self.gearRatio()
        
        return (mtb + stb) / 1000

    def ml2turn(self, value):
        bitRange = 2 ** self.encoderUnit.bits[1] -1
        mtb = round(np.trunc(value / self.pitchVolume() * self.gearRatio() * 1000), 0)
        stb = round(bitRange * (value - self.turn2ml([mtb, 0])) / self.pitchVolume() * self.gearRatio() * 1000, 0)
        return [np.uint32(mtb), np.uint32(stb)]
    
    def value(self, value, range=32):
        rc = (2**range - 1) + value if value < 0 else value
        return rc
    
    def split(self, value, bits, range=32):            
        value = bin(value)[2:].zfill(range)
        return [
            int(value[:bits].zfill(range),2), 
            int(value[bits:].zfill(range),2)
            ]
    
    def merge(self, value, bits, range=32, verbose=False):          
        rc = self.value(int("".join([
            bin(value[0])[2:].zfill(bits), 
            bin(value[1])[2:].zfill(range-bits)]), 2), range)
        return rc

INCS_MAX = 24_185_993

am = AM81111()
am.incs2mulmin(INCS_MAX)


np.float64(2445.612578477266)

In [ ]:
A = np.pow(15,2) * np.pi / 4.

# time in seconds for 10% of the operational range

4.98 / (np.arange(100, 2000, 100) / ( 60 * A))

array([519.54088509, 259.77044254, 173.18029503, 129.88522127,
       103.90817702,  86.59014751,  74.22012644,  64.94261064,
        57.72676501,  51.95408851,  47.23098955,  43.29507376,
        39.96468347,  37.11006322,  34.63605901,  32.47130532,
        30.56122853,  28.8633825 ,  27.34425711])

In [ ]:
A = np.pow(15,2) * np.pi / 4.   # mm²
sp = 5.0                        # mm

mt = 1500 / 2000

mul = sp * A * mt               

mul / 1000.0

np.float64(0.662679700366597)